# import os
os.chdir(r'C:\Users\asama\python\DS 220')
print("Now running in:", os.getcwd())
Project 2 - Data Analysis with Pandas

### Group Member: 
- Chanhyung Kim 
- Abdul Samad 
- Dadi Sriram 
- Fabuluje Victor 
- Sherpa Cia 
- Joshua


### Project Overview
This project analyzes the Nobel laureates dataset to explore trends in award distribution, demographics, and key historical patterns.

### Dataset Description
The dataset used is 'laureates.csv'. 
This dataset contains information about Nobel Prize winners including their names, years of birth/death, categories, and countries.

### Tools and Libraries
- Python
- Pandas
- Matplotlib / Seaborn
- Jupyter Notebook

## Loading and Cleaning the Dataset

In [39]:
import pandas as pd

#Load data file
df = pd.read_csv('laureates.csv')



#Inspect Information
df.head()
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1000 non-null   int64 
 1   firstname          1000 non-null   object
 2   surname            968 non-null    object
 3   born               999 non-null    object
 4   died               1000 non-null   object
 5   bornCountry        969 non-null    object
 6   bornCountryCode    969 non-null    object
 7   bornCity           966 non-null    object
 8   diedCountry        653 non-null    object
 9   diedCountryCode    653 non-null    object
 10  diedCity           647 non-null    object
 11  gender             1000 non-null   object
 12  year               1000 non-null   int64 
 13  category           1000 non-null   object
 14  overallMotivation  23 non-null     object
 15  share              1000 non-null   int64 
 16  motivation         1000 non-null   object
 

id                     0
firstname              0
surname               32
born                   1
died                   0
bornCountry           31
bornCountryCode       31
bornCity              34
diedCountry          347
diedCountryCode      347
diedCity             353
gender                 0
year                   0
category               0
overallMotivation    977
share                  0
motivation             0
name                 264
city                 269
country              267
dtype: int64

In [40]:
#Map camelCase names to clean snake_case
df = df.rename(columns={
    'born':              'birth_date',
    'died':              'death_date',
    'bornCountry':       'born_country',
    'bornCountryCode':   'born_country_code',
    'bornCity':          'born_city',
    'diedCountry':       'died_country',
    'diedCountryCode':   'died_country_code',
    'diedCity':          'died_city',
    'year':              'award_year',
    'overallMotivation': 'overall_motivation'
})

#As a final pass, lowercase everything just in case
df.columns = df.columns.str.lower()


In [41]:
#Turn the birth_date and death_date columns into real datetime objects
df['birth_date'] = pd.to_datetime(df['birth_date'], errors='coerce')
df['death_date'] = pd.to_datetime(df['death_date'], errors='coerce')

#Ensure award_year is an integer bad parses → <NA>
df['award_year'] = pd.to_numeric(df['award_year'], errors='coerce').astype('Int64')


In [42]:
#For Q1 (keywords) and Q3 (age) we need both a motivation text and a birth_date
df = df.dropna(subset=['motivation', 'birth_date'])


In [43]:
#Extract the birth year from the full date
df['birth_year'] = df['birth_date'].dt.year

#Compute age at the time of award
df['age_at_award'] = df['award_year'] - df['birth_year']

#Group awards by decade (e.g., 1994 → 1990s)
df['decade'] = (df['award_year'] // 10) * 10


In [62]:
# Title-case and trim whitespace in 'category'
df['category'] = df['category'].str.title().str.strip()

# Capitalize 'gender' and fill missing values with 'Unknown'
df['gender']   = df['gender'].str.capitalize().fillna('Unknown')


In [45]:
#Lowercase, strip punctuation, trim whitespace
df['motivation_clean'] = (
    df['motivation']
      .str.lower()
      .str.replace(r'[\W_]+', ' ', regex=True)
      .str.strip()
)


In [60]:
#Q1 

from sklearn.feature_extraction.text import CountVectorizer

# Build a term-count matrix, ignoring English stop words
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['motivation_clean'])

# Summarize into a DataFrame of keyword → count
keywords = pd.DataFrame({
    'keyword': vectorizer.get_feature_names_out(),
    'count':   X.sum(axis=0).A1
}).sort_values('count', ascending=False)

display(keywords.head(20))


,keyword,count
579,discovery,198
577,discoveries,163
390,concerning,118
551,development,105
2127,work,97
1958,theory,72
438,contributions,59
1891,structure,55
1652,recognition,47
628,economic,44


In [61]:
#Q7

#Calculate and sort the average age_at_award for each category
avg_age_by_category = df.groupby('category')['age_at_award'].mean().sort_values()

#Display the sorted averages
display(avg_age_by_category)


category
Physics       57.285714
Medicine      58.662222
Chemistry     58.867725
Peace         60.309091
Literature     64.92437
Economics     66.988764
Name: age_at_award, dtype: Float64

In [55]:
# Save the cleaned DataFrame to a CSV file without row indices
df.to_csv('laureates_cleaned.csv', index=False)


In [59]:
important_cols = [
    'motivation_clean',  # Q1
    'decade',            # Q2
    'age_at_award',      # Q3
    'gender',            # Q4
    'born_country',      # Q5
    'award_year',        # Q5 
    'category'           # Q6 & Q7
]
# Create a new, more organized DataFrame with only those columns
df_small = df[important_cols]


#Get overview of the result
df_small.head(15)


,motivation_clean,decade,age_at_award,gender,born_country,award_year,category
0,in recognition of the extraordinary services h...,1900,56,Male,Prussia (now Germany),1901,Physics
1,in recognition of the extraordinary service th...,1900,49,Male,the Netherlands,1902,Physics
2,in recognition of the extraordinary service th...,1900,37,Male,the Netherlands,1902,Physics
3,in recognition of the extraordinary services h...,1900,51,Male,France,1903,Physics
4,in recognition of the extraordinary services t...,1900,44,Male,France,1903,Physics
5,in recognition of the extraordinary services t...,1900,36,Female,Russian Empire (now Poland),1903,Physics
6,in recognition of her services to the advancem...,1910,44,Female,Russian Empire (now Poland),1911,Chemistry
7,for his investigations of the densities of the...,1900,62,Male,United Kingdom,1904,Physics
8,for his work on cathode rays,1900,43,Male,Hungary (now Slovakia),1905,Physics
9,in recognition of the great merits of his theo...,1900,50,Male,United Kingdom,1906,Physics


## Project Questions

#### Question 1 - What are the most frequently mentioned keywords in the prize motivations?

#### Question 2 - During which decades were the most Nobel Prizes awarded?

#### Question 3 - What is the typical age when recipients are awarded the Nobel Prize?

#### Question 4 - How are Nobel Prizes distributed between male and female laureates?

#### Question 5 - Which countries have produced the most Nobel laureates over time?

#### Question 6 - What is the distribution of Nobel Prizes across different categories (e.g., Physics, Peace, Literature)?

#### Question 7 - What is the average age at which laureates receive their Nobel Prize, broken down by category?